# 08 Inference

In [ ]:
!git clone https://github.com/JeserylMae/unet-ensemble.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
sys.path.append('/content/unet-ensemble')

In [ ]:
!pip install segmentation-models-pytorch --quiet

In [ ]:
import torch
from PIL import Image

from src.models.model import (predict,
                              load_model, 
                              visualize_prediction)

In [ ]:
IMAGE_PATH    = "/content/drive/Shareddrives/U-Net Ensemble/Dataset/Evaluation/Normalized/Synthetic/template-a/synthetic-id-template-a-1185.jpg"
OUTPUT_PATH   = "/content/drive/Shareddrives/U-Net Ensemble/Output/mask_output-4.jpg"
THRESHOLD     = 0.5
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ensemble weighting: alpha for U-Net++, (1 - alpha) for Attention U-Net
ALPHA         = 0.5

FROM_HUB      = True
HUB_FILENAME  = "model.safetensors"
HUB_TOKEN     = "read-token"
UNETPP_HUB_REPO  = "Jesayyy/mben-unetpp"
ATTUNET_HUB_REPO = "Jesayyy/mben-attunet"

IMAGE_SIZE = 256

In [ ]:
print(f"Using device: {DEVICE}")

unetpp, attunet = load_model(
    device           = DEVICE,
    from_hub         = FROM_HUB,
    unetpp_hub_repo  = UNETPP_HUB_REPO,
    attunet_hub_repo = ATTUNET_HUB_REPO,
    hub_filename     = HUB_FILENAME,
    hub_token        = HUB_TOKEN,
)

result = predict(
    unetpp     = unetpp,
    attunet    = attunet,
    image_path = IMAGE_PATH,
    image_size = IMAGE_SIZE,
    device     = DEVICE,
    threshold  = THRESHOLD,
    alpha      = ALPHA,          # Y = alpha * UNetPP(X) + (1-alpha) * AttUNet(X)
)

visualize_prediction(
    result,
    output_path   = OUTPUT_PATH,
    colormap      = "jet",        # blue=low -> green=mid -> red=high confidence
    overlay_alpha = 0.55,
    only_masked   = False,
)